# Notebook 1: EXPLORING AND CLEANING OLIST E-COMMERCE DATA
-------------------------------------------------------------------------------------------------

### Author: Emmanuel Aregbesola
### Date: April, 2026



In [1]:
# import pandas library

import pandas as pd

In [2]:
# Load the data

files = {
    'customers' : 'olist_customers_dataset.csv',
    'geolocation' : 'olist_geolocation_dataset.csv',
    'order_items' : 'olist_order_items_dataset.csv',
    'payment' : 'olist_order_payments_dataset.csv',
    'reviews' : 'olist_order_reviews_dataset.csv',
    'orders' : 'olist_orders_dataset.csv',
    'products' : 'olist_products_dataset.csv',
    'sellers' : 'olist_sellers_dataset.csv',
    'product_category' : 'product_category_name_translation.csv'
}

data = {}

for name, file in files.items():
    data[name] = pd.read_csv(file)
    
    
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [3]:
# create placeholders for all dataframe

customers_df = data['customers']
geolocation_df = data['geolocation']
order_items_df = data['order_items']
payments_df = data['payment']
reviews_df = data['reviews']
orders_df = data['orders']
products_df = data['products']
sellers_df = data['sellers']
product_category_df = data['product_category']

In [4]:
# basic exploration function 
def basic_exploration(df):
    # 1. Visual Header
    print(f"{'='*30}")
    print("      VITAL STATISTICS")
    print(f"{'='*30}")
    
    # 2. Dimensions and Red Flags
    print(f"Shape: {df.shape}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print("-" * 30)
    
    # 3. Missing Data Profile (Count and Percentage)
    print("MISSING DATA PROFILE:")
    null_count = df.isna().sum()
    null_percent = (null_count / len(df)) * 100
    null_table = pd.concat([null_count, null_percent.round(2)], axis=1)
    null_table.columns = ['Count', 'Percentage (%)']
    print(null_table)
    print("-" * 30)
    
    # 4. Data Types (Clean View)
    print("DATA TYPES:")
    print(df.dtypes)
    print("-" * 30)
    
    # 5. Summary Statistics (Numeric & Categorical)
    print("SUMMARY STATISTICS:")
    # include='all' ensures we see unique counts for strings, not just math for numbers
    display(df.describe(include='all', datetime_is_numeric=True)) 
    print(f"{'='*30}\n")

## Data Cleaning

This section involves taking each tables individually and cleaning them.

## Payments Dataframe

### order_id: unique identifier of an order.
### payment_sequential: a customer may pay an order with more than one payment method. If he does so, a sequence will be created to
### payment_type: method of payment chosen by the customer.
### payment_installments: number of installments chosen by the customer.
### payment_value: transaction value.

In [5]:
basic_exploration(payments_df)

      VITAL STATISTICS
Shape: (103886, 5)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                      Count  Percentage (%)
order_id                  0             0.0
payment_sequential        0             0.0
payment_type              0             0.0
payment_installments      0             0.0
payment_value             0             0.0
------------------------------
DATA TYPES:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


In [6]:
# check basic overview of payments dataframe

payments_df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [7]:
# Check for frequency for payment type

payments_df['payment_type'].value_counts()

credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: payment_type, dtype: int64

In [8]:
# create a filter dataframe for the not_defined value in the payment_type column

payments_mask = payments_df['payment_type'] == 'not_defined'

payments_df.loc[payments_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [9]:
# check if the order_id for the not_defined values appears in the entire dataframe

problem_id = payments_df.loc[payments_mask, 'order_id']
check_id = payments_df[payments_df['order_id'].isin(problem_id)]
check_id

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [10]:
# drop the not_defined rows

payments_df = payments_df[~payments_mask].reset_index(drop=True)

In [11]:
# validate the drop

payments_df['payment_type'].value_counts()

credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
Name: payment_type, dtype: int64

In [12]:
# filter the dataframe to condition of payment_value = 0

payment_zero_mask = payments_df['payment_value'] == 0.0

payments_df.loc[payment_zero_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
62672,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77883,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
100763,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [13]:
# filter dataframe to condition of payment_installments = 0

installment_zero_mask = payments_df['payment_installments'] == 0

payments_df.loc[installment_zero_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79012,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [14]:
# Fix logical error

payments_df.loc[payments_df['payment_installments'] == 0, 'payment_installments'] = 1

In [15]:
# Validate the payment column

basic_exploration(payments_df)

      VITAL STATISTICS
Shape: (103883, 5)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                      Count  Percentage (%)
order_id                  0             0.0
payment_sequential        0             0.0
payment_type              0             0.0
payment_installments      0             0.0
payment_value             0             0.0
------------------------------
DATA TYPES:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103883,103883.000000,103883,103883.000000,103883.000000
unique,99437,NaN,4,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092681,NaN,2.853422,154.104831
std,NaN,0.706594,NaN,2.687054,217.495628
min,NaN,1.000000,NaN,1.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.800000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.840000


### Summary on payments cleaning

1. After running the basic exploration function on the payments dataframe, it was found that there are no null values, duplicates and the data types are correct. The summary statistics didnt reveal anything out of place except for the fact that there were zero values in the `payment_value`  and `payment_installments` column.            

                   
2. For validity purposes, a value_counts was ran to check what types of payment existed, there it was found that there were `payment_types` tagged `not_defined`. After filtering, it was found that the `not_defined` rows were a sort of ghost records. This was after order_ids that had the `not_defined` values were checked across the dataframe, if there could be another record. They had zero value, and only one payment sequential which led to the rows being dropped.    
Note: These `not_defined` rows were only 3.              

                           
3. We also explored the rows that had zero values. It was found that the `payment_value` that had zero were transactions that were paid with vouchers. Since these are legit transactions, they were not dropped. In the `payment_installments` column, it was concluded that the zer values were logical errors. This is because it is highly unlikely for a transaction with zero installments to have a price. It was decided that the zero values were changed to `1` since the mode of the column is that said value.


                                     
**Rows before cleaning** = 103886     
**Rows before cleaning** = 103883       
**Rows dropped** = 3 

## Orders Dataframe

### order_id: unique identifier of the order.
### customer_id: key to the customer dataset. Each order has a unique customer_id.
### order_status: Reference to the order status (delivered, shipped, etc).
### order_purchase_timestamp: Shows the purchase timestamp.
### order_approved_at: Shows the payment approval timestamp.
### order_delivered_carrier_date: Shows the order posting timestamp. When it was handled to the logistic partner.
### order_delivered_customer_date: Shows the actual order delivery date to the customer.
### order_estimated_delivery_date: Shows the estimated delivery date that was informed to customer at the purchase moment.

In [16]:
# explore the orders dataframe

basic_exploration(orders_df)

      VITAL STATISTICS
Shape: (99441, 8)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                               Count  Percentage (%)
order_id                           0            0.00
customer_id                        0            0.00
order_status                       0            0.00
order_purchase_timestamp           0            0.00
order_approved_at                160            0.16
order_delivered_carrier_date    1783            1.79
order_delivered_customer_date   2965            2.98
order_estimated_delivery_date      0            0.00
------------------------------
DATA TYPES:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
------------------------------
SUM

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [17]:
# convert date columns to datetime

date_columns = ['order_purchase_timestamp',
                'order_approved_at',
                'order_delivered_carrier_date',
                'order_delivered_customer_date',
                'order_estimated_delivery_date'
]

for col in date_columns:
    orders_df[col] = pd.to_datetime(orders_df[col], errors='coerce')

In [18]:
# validate the datetime change

orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [19]:
# checking the value count of order_status in approved transactions

orders_df[orders_df['order_approved_at'].isna()]['order_status'].value_counts()

canceled     141
delivered     14
created        5
Name: order_status, dtype: int64

In [20]:
# creating a dataframe for orders with no approved transaction but are delivered 

approved_delivered = (orders_df['order_status'] == 'delivered') & (orders_df['order_approved_at'].isna())

orders_df.loc[approved_delivered]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


In [21]:
# filling the approved_delivered mask with the purchase_timestamp

orders_df.loc[approved_delivered, 'order_approved_at'] = orders_df.loc[approved_delivered, 'order_purchase_timestamp']

In [22]:
# confirm the fill

approved_delivered = (orders_df['order_status'] == 'delivered') & (orders_df['order_approved_at'].isna())
orders_df[approved_delivered].shape

(0, 8)

In [23]:
# Check delivered orders that have no delievered timestamp

delivered_no_customer = (orders_df['order_status'] == 'delivered') & (orders_df['order_delivered_customer_date'].isna())

orders_df.loc[delivered_no_customer]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [24]:
# check the shape of the mask

orders_df.loc[delivered_no_customer].shape

(8, 8)

In [25]:
# drop the orders that was delivered but have no delivery timestamp

orders_df = orders_df[~delivered_no_customer].reset_index(drop=True)

In [26]:
# check the statuses in order_status

orders_df['order_status'].value_counts()

delivered      96470
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_status, dtype: int64

In [27]:
# Check for canceled orders that has delivered timestamp

canceled_integrity = (orders_df['order_status'] == 'canceled') & (orders_df['order_delivered_customer_date'].notna())

orders_df.loc[canceled_integrity]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8790,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58263,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59329,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92631,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94393,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


In [28]:
# check the shape

orders_df.loc[canceled_integrity].shape

(6, 8)

In [29]:
# drop the canceled integrity table

orders_df = orders_df[~canceled_integrity].reset_index(drop=True)

In [30]:
# validate the delivered orders pathway

bad_delivered = (orders_df['order_status'] == 'delivered') & (
    orders_df['order_approved_at'].isna() | 
    orders_df['order_delivered_carrier_date'].isna() | 
    orders_df['order_delivered_customer_date'].isna()
)

orders_df.loc[bad_delivered]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73215,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14


In [31]:
# Shipped but missing approval/carrier date OR has a customer date already

bad_shipped = (orders_df['order_status'] == 'shipped') & (
    orders_df['order_approved_at'].isna() | 
    orders_df['order_delivered_carrier_date'].isna() | 
    orders_df['order_delivered_customer_date'].notna()
)

orders_df.loc[bad_shipped]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [32]:
# Processing/Invoiced but already has carrier or customer timestamps

bad_preparing = (orders_df['order_status'].isin(['processing', 'invoiced'])) & (
    orders_df['order_delivered_carrier_date'].notna() | 
    orders_df['order_delivered_customer_date'].notna()
)

orders_df.loc[bad_preparing]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [33]:
# Canceled but has a delivery timestamp

bad_canceled = (orders_df['order_status'] == 'canceled') & (
    orders_df['order_delivered_customer_date'].notna()
)

orders_df.loc[bad_canceled]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [34]:
# validate the dataframe

basic_exploration(orders_df)

      VITAL STATISTICS
Shape: (99427, 8)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                               Count  Percentage (%)
order_id                           0            0.00
customer_id                        0            0.00
order_status                       0            0.00
order_purchase_timestamp           0            0.00
order_approved_at                146            0.15
order_delivered_carrier_date    1782            1.79
order_delivered_customer_date   2957            2.97
order_estimated_delivery_date      0            0.00
------------------------------
DATA TYPES:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date   

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99427,99427,99427,99427,99281,97645,96470,99427
unique,99427,99427,8,NaN,NaN,NaN,NaN,NaN
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,NaN,NaN,NaN,NaN,NaN
freq,1,1,96470,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,2017-12-31 09:03:23.721957120,2017-12-31 17:50:33.543639040,2018-01-04 22:07:02.372451072,2018-01-14 12:41:33.581683456,2018-01-24 03:26:44.678809600
min,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00
25%,NaN,NaN,NaN,2017-09-12 15:11:19.500000,2017-09-12 22:15:23,2017-09-15 22:31:52,2017-09-25 22:15:09.500000,2017-10-03 00:00:00
50%,NaN,NaN,NaN,2018-01-18 23:04:36,2018-01-19 10:55:46,2018-01-24 16:09:18,2018-02-02 19:32:21,2018-02-15 00:00:00
75%,NaN,NaN,NaN,2018-05-04 15:29:15.500000,2018-05-04 19:35:19,2018-05-08 13:36:00,2018-05-15 22:54:48.500000,2018-05-25 00:00:00
max,NaN,NaN,NaN,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00


## Summary on Orders Cleaning

1. After the basic exploration of the orders dataframe, it was realized that the date columns in the dataframe were not in the right datatype. The date columns were converted to datetime datatype.

2. It was also discovered that there were null values in the date columns apart from the `order_purchase_timestamp` column. An investigation was conducted;

* First, I investigated the `order_approved_at` column. I checked for the frequency of the order status that had null values in the `order_approved_at` column. The order staus that stood out was the `delivered` status, it would be inaccurate if an order that was delivered wasn't approved in the first place. I concluded that the inaccuracy was due to an entry error and decided to fill the null values with the values in the `order_purchase_timestamp`.

* I went on to investigate the `delivered` status. I filtered the entire dataframe to give me a result, if an order was delivered but that same order did not have a `order_delivered_customer_date`. I decided to drop the filtered dataframe which was 8 rows. The reason i dropped it was to preserve the data integrity of the orders dataframe.
Although, I went on to investigate the pathway of the `delivered`, since it is not meant to have any null value in any of the timestamps. There was a row with a null value in the `order_delivered_carrier_date`, i decided not to drop the row because i assume the seller delivered the order themselves and since there is a `order_delivered_customer_date`.

* This made me check for other statuses that might pose a threat to the integrity of the dataframe. I saw that there were some canceled orders that were delivered, those were also dropped.

## Products Dataframe

### product_id: unique product identifier
### product_category_name: root category of product, in Portuguese.
### product_name_lenght: number of characters extracted from the product name.
### product_description_lenght: number of characters extracted from the product description.
### product_photos_qty: number of product published photos
### product_weight_g: product weight measured in grams.
### product_length_cm: product length measured in centimeters.
### product_height_cm: product height measured in centimeters.
### product_width_cm: product width measured in centimeters.

In [35]:
# explore the products dataframe

basic_exploration(products_df)

      VITAL STATISTICS
Shape: (32951, 9)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                            Count  Percentage (%)
product_id                      0            0.00
product_category_name         610            1.85
product_name_lenght           610            1.85
product_description_lenght    610            1.85
product_photos_qty            610            1.85
product_weight_g                2            0.01
product_length_cm               2            0.01
product_height_cm               2            0.01
product_width_cm                2            0.01
------------------------------
DATA TYPES:
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dty

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32951,32341,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
unique,32951,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,NaN,NaN,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,NaN,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,NaN,NaN,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,NaN,NaN,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,NaN,NaN,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000


In [36]:
# rename columns

products_df.rename(columns=({'product_name_lenght' : 'product_name_length', 
                             'product_description_lenght' : 'product_description_length'}), inplace=True)

In [37]:
# Inspecting the null dataframe

product_null_mask = products_df.isna().any(axis=1)

products_df.loc[product_null_mask].head(50)

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [38]:
# Filling the null values with either 'Unknown' OR '0'

# fill the object column with 'Unknown'
products_df['product_category_name'] = products_df['product_category_name'].fillna('Unknown')

# fill the float column with 0.0
float_col =  ['product_name_length',
             'product_description_length',
             'product_photos_qty',
             'product_weight_g',
             'product_length_cm',
             'product_height_cm',
             'product_width_cm']

products_df[float_col] = products_df[float_col].fillna(0.0)

In [39]:
# confirm the fill

product_null_mask = products_df.isna().any(axis=1)

products_df.loc[product_null_mask].head(50)

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [40]:
# validate the products dataframe

basic_exploration(products_df)

      VITAL STATISTICS
Shape: (32951, 9)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                            Count  Percentage (%)
product_id                      0             0.0
product_category_name           0             0.0
product_name_length             0             0.0
product_description_length      0             0.0
product_photos_qty              0             0.0
product_weight_g                0             0.0
product_length_cm               0             0.0
product_height_cm               0             0.0
product_width_cm                0             0.0
------------------------------
DATA TYPES:
product_id                     object
product_category_name          object
product_name_length           float64
product_description_length    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dty

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32951,32951,32951.000000,32951.000000,32951.000000,32951.000000,32951.000000,32951.000000,32951.000000
unique,32951,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,47.579527,757.213104,2.148463,2276.334315,30.813207,16.936633,23.195320
std,NaN,NaN,12.071951,637.745057,1.745732,4281.945502,16.915648,13.637779,12.080033
min,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,41.000000,326.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,NaN,NaN,51.000000,584.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,NaN,NaN,57.000000,961.000000,3.000000,1900.000000,38.000000,21.000000,30.000000


## Summary of Products Dataframe

1. After the basic exploration, i caught a misspelling in two of the columns in the products dataframe. I renamed them.

2. There were also null values in some the columns in the dataframe. Instead of dropping them, i decided to fill them with `Unknown` and `0.0`. The reason i filled them was because of the primary key `product_id`. This column does not have a null value. If i decide to drop all rows with null values then other dataframes that has to be joined with the primary key in the products dataframe will miss some important data. 

## Order_items Dataframe

### order_id: order unique identifier
### order_item_id: sequential number identifying number of items included in the same order.
### product_id: product unique identifier
### seller_id: seller unique identifier
### shipping_limit_date: Shows the seller shipping limit date for handling the order over to the logistic partner.
### bprice: item price
### freight_value: item freight value item (if an order has more than one item the freight value is splitted between items)

In [41]:
# explore the order_items dataframe

basic_exploration(order_items_df)

      VITAL STATISTICS
Shape: (112650, 7)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                     Count  Percentage (%)
order_id                 0             0.0
order_item_id            0             0.0
product_id               0             0.0
seller_id                0             0.0
shipping_limit_date      0             0.0
price                    0             0.0
freight_value            0             0.0
------------------------------
DATA TYPES:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,112650,112650.000000,112650,112650,112650,112650.000000,112650.000000
unique,98666,NaN,32951,3095,93318,NaN,NaN
top,8272b63d03f5f79c56e9e4120aec44ef,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,6560211a19b47992c3666cc44a7e94c0,2017-07-21 18:25:23,NaN,NaN
freq,21,NaN,527,2033,21,NaN,NaN
mean,NaN,1.197834,NaN,NaN,NaN,120.653739,19.990320
std,NaN,0.705124,NaN,NaN,NaN,183.633928,15.806405
min,NaN,1.000000,NaN,NaN,NaN,0.850000,0.000000
25%,NaN,1.000000,NaN,NaN,NaN,39.900000,13.080000
50%,NaN,1.000000,NaN,NaN,NaN,74.990000,16.260000
75%,NaN,1.000000,NaN,NaN,NaN,134.900000,21.150000


In [42]:
# convert date column to datetime

order_items_df['shipping_limit_date'] = pd.to_datetime(order_items_df['shipping_limit_date'])

In [43]:
# price lesser than R$ 1

price_mask = order_items_df['price'] < 1

order_items_df.loc[price_mask]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
27652,3ee6513ae7ea23bdfab5b9ab60bffcb5,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,2018-05-04 03:55:26,0.85,18.23
48625,6e864b3f0ec71031117ad4cf46b7f2a1,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,2018-05-02 20:30:34,0.85,18.23
87081,c5bdd8ef3c0ec420232e668302179113,2,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,2018-05-07 02:55:22,0.85,22.30


In [44]:
# freight value = 0

freight_mask = order_items_df['freight_value'] == 0

order_items_df.loc[freight_mask]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
...,...,...,...,...,...,...,...
111094,fc698f330ec7fb74859071cc6cb29772,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-04-25 02:31:57,53.9,0.0
111497,fd4907109f6bac23f07064af84bec02d,1,7a10781637204d8d10485c71a6108a2e,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-30 11:31:32,219.0,0.0
111649,fd95e4b85ebbb81853d4a6be3d61432b,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-05-04 11:10:31,106.9,0.0
112182,fee19a0dc7358b6962a611cecf6a37b4,1,f1c7f353075ce59d8a6f3cf58f419c9c,37be5a7c751166fbc5f8ccba4119e043,2017-09-07 22:06:31,195.0,0.0


In [45]:
# validate the dataframe

basic_exploration(order_items_df)

      VITAL STATISTICS
Shape: (112650, 7)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                     Count  Percentage (%)
order_id                 0             0.0
order_item_id            0             0.0
product_id               0             0.0
seller_id                0             0.0
shipping_limit_date      0             0.0
price                    0             0.0
freight_value            0             0.0
------------------------------
DATA TYPES:
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,112650,112650.000000,112650,112650,112650,112650.000000,112650.000000
unique,98666,NaN,32951,3095,NaN,NaN,NaN
top,8272b63d03f5f79c56e9e4120aec44ef,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,6560211a19b47992c3666cc44a7e94c0,NaN,NaN,NaN
freq,21,NaN,527,2033,NaN,NaN,NaN
mean,NaN,1.197834,NaN,NaN,2018-01-07 15:36:52.192685312,120.653739,19.990320
min,NaN,1.000000,NaN,NaN,2016-09-19 00:15:34,0.850000,0.000000
25%,NaN,1.000000,NaN,NaN,2017-09-20 20:57:27.500000,39.900000,13.080000
50%,NaN,1.000000,NaN,NaN,2018-01-26 13:59:35,74.990000,16.260000
75%,NaN,1.000000,NaN,NaN,2018-05-10 14:34:00.750000128,134.900000,21.150000
max,NaN,21.000000,NaN,NaN,2020-04-09 22:35:08,6735.000000,409.680000


## Summary of Order_items Dataframe

1. After the basic exploration, i noticed that the date column, `shipping_limit_date` has a wrong data type. I changed the data type to datetime.

2. Also, i noticed that for the `price` column the minimum was `0.85` and for the `freight value` the minimum was `0`. After investigation, i concluded that these were legit rows of data. For the `price` column, the `0.85` price was for a particular product that also had different order_id, that means it was sold to different customers. Also, the transactions happened at different time according to the timestamps. For the `freight_value` column, I concluded that the value was 0 due to discount or some promotional event.

## Reviews Dataframe

### review_id: unique review identifier
### order_id: unique order identifier
### review_score: Note ranging from 1 to 5 given by the customer on a satisfaction survey.
### review_comment_title: Comment title from the review left by the customer, in Portuguese.
### review_comment_message: Comment message from the review left by the customer, in Portuguese.
### review_creation_date: Shows the date in which the satisfaction survey was sent to the customer.
### review_answer_timestamp: Shows satisfaction survey answer timestamp.

In [46]:
# explore the dataframe

basic_exploration(reviews_df)

      VITAL STATISTICS
Shape: (99224, 7)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                         Count  Percentage (%)
review_id                    0            0.00
order_id                     0            0.00
review_score                 0            0.00
review_comment_title     87656           88.34
review_comment_message   58247           58.70
review_creation_date         0            0.00
review_answer_timestamp      0            0.00
------------------------------
DATA TYPES:
review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object
------------------------------
SUMMARY STATISTICS:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
count,99224,99224,99224.000000,11568,40977,99224,99224
unique,98410,98673,NaN,4527,36159,636,98248
top,7b606b0d57b078384f0b58eac1d41d78,c88b1d1b157a9999ce368f218a407141,NaN,Recomendo,Muito bom,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,NaN,423,230,463,4
mean,NaN,NaN,4.086421,NaN,NaN,NaN,NaN
std,NaN,NaN,1.347579,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,4.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN


In [47]:
# check if review_score fall into expexted 1-5 range

reviews_df['review_score'].value_counts()

5    57328
4    19142
1    11424
3     8179
2     3151
Name: review_score, dtype: int64

In [48]:
# feature engineering for review_comment_title and review_comment_message

reviews_df['has_title'] = reviews_df['review_comment_title'].notna().astype(int)
reviews_df['has_comment'] = reviews_df['review_comment_message'].notna().astype(int)

In [49]:
# convert date columns to datetime

date_cols = ['review_creation_date', 'review_answer_timestamp']

for col in date_cols:
    reviews_df[col] = pd.to_datetime(reviews_df[col], errors='coerce')

In [50]:
# check duplicates in review_id

reviews_df[reviews_df['review_id'].duplicated()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_title,has_comment
3317,3242cc306a9218d0377831e175d62fbf,9c5bfba7de6a4abbb6ba0baab78d1622,5,NaN,Ótimo,2017-07-21,2017-07-22 01:06:37,0,1
5719,308316408775d1600dad81bd3184556d,3fe4dbcdb046a475dbf25463c1ca78bd,5,NaN,Ajudem a rastrear uma das 3 prateleiras que fa...,2017-09-07,2017-09-11 09:58:09,0,1
7213,8ee90ac383cf825bb7f4756130d4e74a,75d5d3d16567a27eefc5752aeb063072,5,NaN,recomendo,2017-05-30,2017-06-02 18:16:17,0,1
7805,d433c252647c51309432ca0b763f969b,191abde9c3bf77d1ce67d100d8db6d15,1,NaN,NaN,2017-09-03,2017-09-06 16:11:00,0,0
9013,03a6a25db577d0689440933055111897,3fde8b7313af6b37b84b5c7594d7add0,5,NaN,Muito Bom! Gostei Bastante! Tecido Ótimo! Aten...,2017-12-15,2017-12-16 01:32:18,0,1
...,...,...,...,...,...,...,...,...,...
99108,2c6c08892b83ba4c1be33037c2842294,42ae1967f68c90bb325783ac55d761ce,4,NaN,"Chegou um pouco amassada, mas nada de mais, e ...",2017-07-03,2017-07-05 19:06:59,0,1
99124,6ec93e77f444e0b1703740a69122e35d,e1fdc6e9d1ca132377e862593a7c0bd4,5,NaN,Vendedor compromisso do vou o cliente,2017-10-07,2017-10-07 19:47:11,0,1
99164,2afe63a67dfd99b3038f568fb47ee761,c5334d330e36d2a810a7a13c72e135ee,5,NaN,"Muito bom, produto conforme anunciado, entrega...",2018-03-03,2018-03-04 22:56:47,0,1
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30,0,0


In [51]:
# check the shape of duplicated values

reviews_df[reviews_df['review_id'].duplicated()].shape

(814, 9)

In [52]:
# drop the duplicate but keep the updated reviews

reviews_df = reviews_df.drop_duplicates(subset='review_id', keep='last')

In [53]:
# confirm the drop

reviews_df[reviews_df['review_id'].duplicated()].shape

(0, 9)

In [54]:
# validate the dataframe

basic_exploration(reviews_df)

      VITAL STATISTICS
Shape: (98410, 9)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                         Count  Percentage (%)
review_id                    0            0.00
order_id                     0            0.00
review_score                 0            0.00
review_comment_title     86891           88.29
review_comment_message   57742           58.67
review_creation_date         0            0.00
review_answer_timestamp      0            0.00
has_title                    0            0.00
has_comment                  0            0.00
------------------------------
DATA TYPES:
review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
has_title                           int32
has_comment                         int

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_title,has_comment
count,98410,98410,98410.000000,11519,40668,98410,98410,98410.000000,98410.000000
unique,98410,98128,NaN,4527,36159,NaN,NaN,NaN,NaN
top,7bc2406110b926393aa56f80a40eba40,8e17072ec97ce29f0e1f111e598b0c85,NaN,Recomendo,Muito bom,NaN,NaN,NaN,NaN
freq,1,3,NaN,422,229,NaN,NaN,NaN,NaN
mean,NaN,NaN,4.088802,NaN,NaN,2018-01-13 06:50:35.984148224,2018-01-16 10:26:43.416786944,0.117051,0.413251
min,NaN,NaN,1.000000,NaN,NaN,2016-10-02 00:00:00,2016-10-07 18:32:28,0.000000,0.000000
25%,NaN,NaN,4.000000,NaN,NaN,2017-09-24 00:00:00,2017-09-27 19:43:43.500000,0.000000,0.000000
50%,NaN,NaN,5.000000,NaN,NaN,2018-02-02 00:00:00,2018-02-05 11:08:52,0.000000,0.000000
75%,NaN,NaN,5.000000,NaN,NaN,2018-05-16 00:00:00,2018-05-20 14:08:15.750000128,0.000000,1.000000
max,NaN,NaN,5.000000,NaN,NaN,2018-08-31 00:00:00,2018-10-29 12:27:35,1.000000,1.000000


## Summary on reviews dataframe

1. After the basic_exploration, I decided to do a frequency count on the review_score column to make sure that all the reviews are between the range 1-5, as any value below or above that range would be consider a data or entry error.

2. I decided to do a feature engineering for `review_comment_title` and `review_comment_message`. The reason being that, both columns have trendmendous high amount of null values. This way i can isolate the reviews that has value and do some analysis on them.

3. I converted the date columns to datetime datatypes

4. From the exploration i noticed that, some of the review_id were not unique. I ran a duplicate check and concluded that the duplicates were updated reviews. I decided to drop all duplicates but keep the updated reviews.
**Duplicate rows dropped:** 814

## customers dataframe

### customer_id: key to the orders dataset. Each order has a unique customer_id
### customer_unique_id: unique identifier of a customer.
### customer_zip_code_prefix: first five digits of customer zip code
### customer_city: customer city name
### customer_state: customer state

In [55]:
# explore the customer dataframe

basic_exploration(customers_df)

      VITAL STATISTICS
Shape: (99441, 5)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                          Count  Percentage (%)
customer_id                   0             0.0
customer_unique_id            0             0.0
customer_zip_code_prefix      0             0.0
customer_city                 0             0.0
customer_state                0             0.0
------------------------------
DATA TYPES:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
------------------------------
SUMMARY STATISTICS:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


## Summary on customers dataframe

In the describe section of the basic exploration, The customer_id are unique, there are no duplicates. The datatypes are correct and there are no null values.      
The customers dataframe is clean.

## Sellers dataframe

### seller_id: seller unique identifier
### seller_zip_code_prefix: first 5 digits of seller zip code
### seller_city: seller city name
### seller_state: seller state

In [56]:
# explore the sellers dataframe

basic_exploration(sellers_df)

      VITAL STATISTICS
Shape: (3095, 4)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                        Count  Percentage (%)
seller_id                   0             0.0
seller_zip_code_prefix      0             0.0
seller_city                 0             0.0
seller_state                0             0.0
------------------------------
DATA TYPES:
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object
------------------------------
SUMMARY STATISTICS:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
count,3095,3095.000000,3095,3095
unique,3095,NaN,611,23
top,3442f8959a84dea7ee197c632cb2df15,NaN,sao paulo,SP
freq,1,NaN,694,1849
mean,NaN,32291.059451,NaN,NaN
std,NaN,32713.453830,NaN,NaN
min,NaN,1001.000000,NaN,NaN
25%,NaN,7093.500000,NaN,NaN
50%,NaN,14940.000000,NaN,NaN
75%,NaN,64552.500000,NaN,NaN


## Summary of sellers dataframe

The seller_id column is unique, there are no duplicates. Also, no null values and the datatypes are correct.    
Sellers dataframe is clean.      
Also, the `seller_zip_code_prefix` is between `1001` and `99730` which is accurate for brazillian zip code prefix.

## Geolocation dataframe

### geolocation_zip_code_prefix: first 5 digits of zip code
### geolocation_lat: latitude
### geolocation_lng: longitude
### geolocation_city: city name
### geolocation_state: state

In [57]:
# explore the geolocation dataframe

basic_exploration(geolocation_df)

      VITAL STATISTICS
Shape: (1000163, 5)
Duplicate Rows: 261831
------------------------------
MISSING DATA PROFILE:
                             Count  Percentage (%)
geolocation_zip_code_prefix      0             0.0
geolocation_lat                  0             0.0
geolocation_lng                  0             0.0
geolocation_city                 0             0.0
geolocation_state                0             0.0
------------------------------
DATA TYPES:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object
------------------------------
SUMMARY STATISTICS:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
count,1.000163e+06,1.000163e+06,1.000163e+06,1000163,1000163
unique,NaN,NaN,NaN,8011,27
top,NaN,NaN,NaN,sao paulo,SP
freq,NaN,NaN,NaN,135800,404268
mean,3.657417e+04,-2.117615e+01,-4.639054e+01,NaN,NaN
std,3.054934e+04,5.715866e+00,4.269748e+00,NaN,NaN
min,1.001000e+03,-3.660537e+01,-1.014668e+02,NaN,NaN
25%,1.107500e+04,-2.360355e+01,-4.857317e+01,NaN,NaN
50%,2.653000e+04,-2.291938e+01,-4.663788e+01,NaN,NaN
75%,6.350400e+04,-1.997962e+01,-4.376771e+01,NaN,NaN


In [58]:
# aggregate the dataframe to one unique row per zip code

geolocation_df = geolocation_df.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'median',
    'geolocation_lng': 'median',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

## Geolocation dataframe

After running the basic exploration function, i saw that there were over 200k duplicates. I decided to investigate. What i found out was that, there were multiple gps coordinates per zip code prefix. To prevent duplicating order records in future joins, i decided to aggregate the dataframe to one unique row per zip code.

In doing that, I aggregated the longitude and latitude using median to get the center point and prevent choosing an outlier of some sort. Then I chose the first city and state for the zip code to use for the aggregated row.

## Product_category Dataframe

### product_category_name: category name in Portuguese
### product_category_name_english: category name in English

In [59]:
# explore the product_category dataframe

basic_exploration(product_category_df)

      VITAL STATISTICS
Shape: (71, 2)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                               Count  Percentage (%)
product_category_name              0             0.0
product_category_name_english      0             0.0
------------------------------
DATA TYPES:
product_category_name            object
product_category_name_english    object
dtype: object
------------------------------
SUMMARY STATISTICS:


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


## summary of product_category dataframe

This dataframe is clean! There are no null values or duplicates.

# Refrential Integrity

This section helps check if the relationship between our table holds up or there are data that points to nothing.

In [60]:
# payments that does not have and id in the orders' table

mismatched_payments = ~payments_df['order_id'].isin(orders_df['order_id'])
payments_df.loc[mismatched_payments].shape

(14, 5)

In [61]:
# drop the mismatched records

payments_df = payments_df[~mismatched_payments].reset_index(drop=True)

In [62]:
payments_df

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103864,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103865,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103866,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103867,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [63]:
# items not matching to an order_id

mismatched_items = ~order_items_df['order_id'].isin(orders_df['order_id'])
order_items_df.loc[mismatched_items].shape

(15, 7)

In [64]:
# dropped the mismatched items

order_items_df = order_items_df[~mismatched_items].reset_index(drop=True)

In [65]:
order_items_df

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112630,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112631,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112632,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112633,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [67]:
# orders that cannot be pointed to a customer

mismatched_customers = ~orders_df['customer_id'].isin(customers_df['customer_id'])
orders_df.loc[mismatched_customers].shape

(0, 8)

In [69]:
# items that doesn't exist in the product dataframe

mismatched_products = ~order_items_df['product_id'].isin(products_df['product_id'])
order_items_df.loc[mismatched_products].shape

(0, 7)

In [70]:
# items sold that does not have a seller

mismatched_sellers = ~order_items_df['seller_id'].isin(sellers_df['seller_id'])
order_items_df.loc[mismatched_sellers].shape

(0, 7)

In [76]:
# sellers whose location are mismatched 

mismatched_geo = ~sellers_df['seller_zip_code_prefix'].isin(geolocation_df['geolocation_zip_code_prefix'])
sellers_df.loc[mismatched_geo].shape

(7, 4)

In [77]:
# customers whose location are mismatched

mismatched_location = ~customers_df['customer_zip_code_prefix'].isin(geolocation_df['geolocation_zip_code_prefix'])
customers_df.loc[mismatched_location].shape

(278, 5)

## Data Integrity and Refrential Integrity Audit

Before proceeding to the exploratory analysis phase, a thorough audit was done to ensure logical consistency across the relational schema.

1. **Payment Vs Orders Audit**: 14 orders existed in the `payments` table without a corresponding entry in the `orders` table which is the master table. These records were tagged as ghost records and thereby was removed from the `payments` tables to ensure intgrity. Without an associated orders, these payments lack timestamps, customer data, product context, making them analytically useless.          

2. **Order_items Vs Orders Audit**: 15 orders existed in the `payments` tables without a corresponding entry in the `orders` table. These records were also tagged ghost records and thereby was removed/dropped to ensure data integrity. Without a corresponding order, these items will lack payment info, customer data, timestamp etc, making them analytically useless also.                

3. **Customers/Sellers Vs Geolocation Audit**: 7 sellers and 278 customers have zip_code_prefixes that does not exist in the clean master geolocation table. I decided to keep the rows so as to preserve financial integrity. If i have dropped the rows of which the sellers and customers were missing, i would have orphaned some rows in the other tables. The steps too take when performing analysis is to perform a left join.               


In [78]:
# Save all cleaned data into a csv file

cleaned_data = {
    'customers': customers_df,
    'geolocation': geolocation_df,
    'order_items': order_items_df,
    'payment': payments_df,
    'reviews': reviews_df,
    'orders': orders_df,
    'products': products_df,
    'sellers': sellers_df,
    'product_category': product_category_df
}


for name, df in cleaned_data.items():
    df.to_csv(f'cleaned_{name}.csv', index=False)
    print(f"✓ Saved: cleaned_{name}.csv ({len(df)} rows)")
    
print(f'All cleaned dataframe are saved!')

✓ Saved: cleaned_customers.csv (99441 rows)
✓ Saved: cleaned_geolocation.csv (19015 rows)
✓ Saved: cleaned_order_items.csv (112635 rows)
✓ Saved: cleaned_payment.csv (103869 rows)
✓ Saved: cleaned_reviews.csv (98410 rows)
✓ Saved: cleaned_orders.csv (99427 rows)
✓ Saved: cleaned_products.csv (32951 rows)
✓ Saved: cleaned_sellers.csv (3095 rows)
✓ Saved: cleaned_product_category.csv (71 rows)
All cleaned dataframe are saved!
